# Task 3 - Netflix Recommendation System Analysis

## Objective

The objective of this task is to build a basic content-based recommendation system using the Netflix dataset.

The system will recommend similar Netflix titles based on available content information such as title, genre, country, director, rating, and content type.

## Approach

The recommendation system will use:

- Text preprocessing
- TF-IDF Vectorization
- Cosine Similarity
- Content-based filtering

## Task Workflow

1. Extract relevant content features
2. Perform text preprocessing
3. Calculate content similarity
4. Generate recommendations for selected titles
5. Evaluate recommendation quality

## Tools Used

- Python
- Pandas
- NumPy
- Scikit-learn
- Jupyter Notebook

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../Task_2_EDA/cleaned_dataset.csv")
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (8790, 16)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,added_year,added_month,added_day,duration_value,duration_type,primary_genre
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,25,90,min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,24,1,Seasons,Crime TV Shows
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,24,1,Seasons,TV Dramas
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,22,91,min,Children & Family Movies
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,24,125,min,Dramas


In [3]:
recommendation_df = df[
    [
        "show_id",
        "title",
        "type",
        "director",
        "country",
        "rating",
        "listed_in",
        "primary_genre"
    ]
].copy()

recommendation_df.head()

,show_id,title,type,director,country,rating,listed_in,primary_genre
0,s1,Dick Johnson Is Dead,Movie,Kirsten Johnson,United States,PG-13,Documentaries,Documentaries
1,s3,Ganglands,TV Show,Julien Leclercq,France,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",Crime TV Shows
2,s6,Midnight Mass,TV Show,Mike Flanagan,United States,TV-MA,"TV Dramas, TV Horror, TV Mysteries",TV Dramas
3,s14,Confessions of an Invisible Girl,Movie,Bruno Garotti,Brazil,TV-PG,"Children & Family Movies, Comedies",Children & Family Movies
4,s8,Sankofa,Movie,Haile Gerima,United States,TV-MA,"Dramas, Independent Movies, International Movies",Dramas


In [4]:
text_columns = ["title", "type", "director", "country", "rating", "listed_in", "primary_genre"]

print(recommendation_df[text_columns].isnull().sum())

title            0
type             0
director         0
country          0
rating           0
listed_in        0
primary_genre    0
dtype: int64


In [5]:
recommendation_df["combined_features"] = (
    recommendation_df["type"].fillna("") + " " +
    recommendation_df["director"].fillna("") + " " +
    recommendation_df["country"].fillna("") + " " +
    recommendation_df["rating"].fillna("") + " " +
    recommendation_df["listed_in"].fillna("") + " " +
    recommendation_df["primary_genre"].fillna("")
)

recommendation_df[
    ["title", "combined_features"]
].head()

,title,combined_features
0,Dick Johnson Is Dead,Movie Kirsten Johnson United States PG-13 Docu...
1,Ganglands,TV Show Julien Leclercq France TV-MA Crime TV ...
2,Midnight Mass,TV Show Mike Flanagan United States TV-MA TV D...
3,Confessions of an Invisible Girl,Movie Bruno Garotti Brazil TV-PG Children & Fa...
4,Sankofa,"Movie Haile Gerima United States TV-MA Dramas,..."


In [6]:
recommendation_df["combined_features"] = (
    recommendation_df["combined_features"]
    .str.lower()
    .str.replace(",", " ", regex=False)
    .str.replace("&", " ", regex=False)
    .str.replace("  ", " ", regex=False)
    .str.strip()
)

recommendation_df[
    ["title", "combined_features"]
].head()

,title,combined_features
0,Dick Johnson Is Dead,movie kirsten johnson united states pg-13 docu...
1,Ganglands,tv show julien leclercq france tv-ma crime tv ...
2,Midnight Mass,tv show mike flanagan united states tv-ma tv d...
3,Confessions of an Invisible Girl,movie bruno garotti brazil tv-pg children fam...
4,Sankofa,movie haile gerima united states tv-ma dramas ...


In [7]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(recommendation_df["combined_features"])
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (8790, 6569)


In [8]:
cosine_sim = cosine_similarity(tfidf_matrix)
print("Similarity Matrix Shape:", cosine_sim.shape)

Similarity Matrix Shape: (8790, 8790)


# Recommendation Function

In [9]:
def recommend_titles(title, top_n=10):
    title_matches = recommendation_df[
        recommendation_df["title"].str.lower() == title.lower()
    ]
    
    if title_matches.empty:
        print(f"Title '{title}' not found in the dataset.")
        return pd.DataFrame()
    
    title_index = title_matches.index[0]
    
    similarity_scores = list(enumerate(cosine_sim[title_index]))
    
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    similar_titles = similarity_scores[1: top_n + 1]
    
    recommended_indices = [index for index, score in similar_titles]
    recommended_scores = [score for index, score in similar_titles]
    
    recommendations = recommendation_df.iloc[recommended_indices][
        ["title", "type", "primary_genre", "rating"]
    ].copy()
    
    recommendations["similarity_score"] = np.round(recommended_scores, 3)
    
    return recommendations.reset_index(drop=True)

In [10]:
recommend_titles("Midnight Mass")

,title,type,primary_genre,rating,similarity_score
0,Gerald's Game,Movie,Horror Movies,TV-MA,0.724
1,Brand New Cherry Flavor,TV Show,TV Dramas,TV-MA,0.717
2,The Haunting of Bly Manor,TV Show,TV Dramas,TV-MA,0.717
3,Ratched,TV Show,TV Dramas,TV-MA,0.717
4,The Haunting of Hill House,TV Show,TV Dramas,TV-MA,0.717
5,The Originals,TV Show,TV Dramas,TV-14,0.690
6,Hush,Movie,Horror Movies,R,0.675
7,The Mist,TV Show,TV Dramas,TV-MA,0.646
8,Before I Wake,Movie,Horror Movies,PG-13,0.642
9,Tiny Pretty Things,TV Show,TV Dramas,TV-MA,0.634


In [11]:
recommend_titles("Sankofa")

,title,type,primary_genre,rating,similarity_score
0,Residue,Movie,Dramas,TV-MA,0.586
1,Seven (Tamil),Movie,Dramas,TV-MA,0.319
2,BoJack Horseman Christmas Special: Sabrina's C...,Movie,Movies,TV-MA,0.312
3,Last Summer,Movie,Dramas,TV-MA,0.304
4,The Light of My Eyes,Movie,Dramas,TV-MA,0.304
5,Taramani,Movie,Dramas,TV-MA,0.300
6,Pose,TV Show,TV Dramas,TV-MA,0.298
7,The Queen's Gambit,TV Show,TV Dramas,TV-MA,0.298
8,Hollywood,TV Show,TV Dramas,TV-MA,0.298
9,Self Made: Inspired by the Life of Madam C.J. ...,TV Show,TV Dramas,TV-MA,0.298


In [12]:
recommend_titles("Ganglands")

,title,type,primary_genre,rating,similarity_score
0,Lupin,TV Show,Crime TV Shows,TV-MA,0.781
1,Crime Time,TV Show,Crime TV Shows,TV-MA,0.781
2,Mortel,TV Show,Crime TV Shows,TV-MA,0.738
3,Ad Vitam,TV Show,Crime TV Shows,TV-MA,0.738
4,Black Spot,TV Show,Crime TV Shows,TV-MA,0.738
5,The Chalet,TV Show,Crime TV Shows,TV-MA,0.738
6,The Frozen Dead,TV Show,Crime TV Shows,TV-MA,0.738
7,La Mante,TV Show,Crime TV Shows,TV-MA,0.738
8,Cannabis,TV Show,Crime TV Shows,TV-MA,0.738
9,Taken,TV Show,Crime TV Shows,TV-14,0.721


In [16]:
def evaluate_recommendations(title, top_n=10):
    recommendations = recommend_titles(title, top_n)
    
    if recommendations.empty:
        return
    
    source = recommendation_df[
        recommendation_df["title"].str.lower() == title.lower()
    ].iloc[0]
    
    source_genre = source["primary_genre"]
    source_type = source["type"]
    
    recommendations["same_genre"] = (
        recommendations["primary_genre"] == source_genre
    )
    
    recommendations["same_type"] = (
        recommendations["type"] == source_type
    )
    
    genre_match_rate = recommendations["same_genre"].mean() * 100
    type_match_rate = recommendations["same_type"].mean() * 100
    
    print(f"Source Title: {title}")
    print(f"Primary Genre: {source_genre}")
    print(f"Content Type: {source_type}")
    print(f"Same Genre Recommendation Rate: {genre_match_rate:.2f}%")
    print(f"Same Content Type Rate: {type_match_rate:.2f}%")
    
    return recommendations

In [17]:
evaluation_results = evaluate_recommendations("Midnight Mass")
evaluation_results

Source Title: Midnight Mass
Primary Genre: TV Dramas
Content Type: TV Show
Same Genre Recommendation Rate: 70.00%
Same Content Type Rate: 70.00%


,title,type,primary_genre,rating,similarity_score,same_genre,same_type
0,Gerald's Game,Movie,Horror Movies,TV-MA,0.724,False,False
1,Brand New Cherry Flavor,TV Show,TV Dramas,TV-MA,0.717,True,True
2,The Haunting of Bly Manor,TV Show,TV Dramas,TV-MA,0.717,True,True
3,Ratched,TV Show,TV Dramas,TV-MA,0.717,True,True
4,The Haunting of Hill House,TV Show,TV Dramas,TV-MA,0.717,True,True
5,The Originals,TV Show,TV Dramas,TV-14,0.690,True,True
6,Hush,Movie,Horror Movies,R,0.675,False,False
7,The Mist,TV Show,TV Dramas,TV-MA,0.646,True,True
8,Before I Wake,Movie,Horror Movies,PG-13,0.642,False,False
9,Tiny Pretty Things,TV Show,TV Dramas,TV-MA,0.634,True,True


In [18]:
evaluation_results = evaluate_recommendations("Ganglands")
evaluation_results

Source Title: Ganglands
Primary Genre: Crime TV Shows
Content Type: TV Show
Same Genre Recommendation Rate: 100.00%
Same Content Type Rate: 100.00%


,title,type,primary_genre,rating,similarity_score,same_genre,same_type
0,Lupin,TV Show,Crime TV Shows,TV-MA,0.781,True,True
1,Crime Time,TV Show,Crime TV Shows,TV-MA,0.781,True,True
2,Mortel,TV Show,Crime TV Shows,TV-MA,0.738,True,True
3,Ad Vitam,TV Show,Crime TV Shows,TV-MA,0.738,True,True
4,Black Spot,TV Show,Crime TV Shows,TV-MA,0.738,True,True
5,The Chalet,TV Show,Crime TV Shows,TV-MA,0.738,True,True
6,The Frozen Dead,TV Show,Crime TV Shows,TV-MA,0.738,True,True
7,La Mante,TV Show,Crime TV Shows,TV-MA,0.738,True,True
8,Cannabis,TV Show,Crime TV Shows,TV-MA,0.738,True,True
9,Taken,TV Show,Crime TV Shows,TV-14,0.721,True,True


## Task 3 Conclusion

A content-based recommendation system was developed using the Netflix dataset.

TF-IDF was used to convert the combined content features into numerical vectors, and cosine similarity was used to identify similar titles.

The recommendation system was tested on different Netflix titles. For "Midnight Mass", 70% of the top 10 recommendations matched the same primary genre and 70% matched the same content type. For "Ganglands", both genre and content type match rates were 100%.

The results show that the system can generate relevant recommendations based on the available content features.

### Limitations

This is a content-based recommendation system, so it does not use user ratings, watch history, or user preferences. The evaluation was also based on genre and content-type consistency because the dataset does not contain actual user interaction data.